# Publish Sample Data for V2 OData Query Examples

This notebook is the V2 companion of `../../query/publish_sample_data.ipynb`. It creates structured sample data in NI Measurement Data Services (V2 mode) that the V2 query notebook uses.

**Run this notebook first** before using `query_measurements.ipynb`.

## What This Creates

This notebook sets up a realistic test scenario with:
- **3 TestResults**: 2 Power Supply tests and 1 Audio Amplifier test
- **12 Test Steps**: Various measurement procedures across all tests
- **38 Measurements**: Scalar measurements (voltages, currents, power, THD, etc.)
- **Conditions on every step**: Environmental and setup parameters. In V2, conditions belong to the step, and text values are supported.
- **Metadata**: 3 operators, 3 test stations, 2 hardware items, 2 UUTs, 3 UUT instances, 3 software items
- **Aliases**: Human-readable references for all metadata entities

## Prerequisites

- NI Measurement Data Services running in V2 mode
- Python environment with the `ni.datastore` package (see `../README.md`)

In [ ]:
# Perform example-specific setup. This is not needed when writing production code.
# Set MDS_V2_GRPC_TARGET (for example, "[::1]:59606") to connect to a specific V2 service.
# When it is not set, the clients use discovery to find the service.
import os

import grpc

grpc_target = os.environ.get("MDS_V2_GRPC_TARGET", "")
# Remove gRPC's 4 MB receive limit so that large values, such as long waveforms, can be read.
grpc_options = [("grpc.max_receive_message_length", -1)]
grpc_channel = grpc.insecure_channel(grpc_target, options=grpc_options) if grpc_target else None

from ni.datastore.metadata import (
    MetadataStoreClient,
    HardwareItem,
    Operator,
    SoftwareItem,
    TestStation,
    Uut,
    UutInstance,
)
from ni.datastore.data.v2 import DataStoreClient

# Create clients
metadata_store_client = MetadataStoreClient(grpc_channel=grpc_channel)
data_store_client = DataStoreClient(grpc_channel=grpc_channel)

## Create Metadata Entities

Setting up the metadata for operators, test stations, hardware, etc. This is the same as in V1, because V2 reuses the V1 Metadata API. This is typically done once and establishes 'aliases' which are human-readable strings that can be used to refer to metadata later.

In [ ]:
print("🚀 Starting sample data creation...")

# Create sample operators
print("\n✅ Creating operators...")
operator1 = Operator(name="Alex Smith", role="Test Engineer")
metadata_store_client.create_operator(operator1)
ALIAS_OPERATOR_ALEX = "V2_Operator_Alex_Smith"
metadata_store_client.create_alias(ALIAS_OPERATOR_ALEX, operator1)

operator2 = Operator(name="Jordan Chen", role="Senior Test Engineer")
metadata_store_client.create_operator(operator2)
ALIAS_OPERATOR_JORDAN = "V2_Operator_Jordan_Chen"
metadata_store_client.create_alias(ALIAS_OPERATOR_JORDAN, operator2)

operator3 = Operator(name="Taylor Johnson", role="Lab Technician")
metadata_store_client.create_operator(operator3)
ALIAS_OPERATOR_TAYLOR = "V2_Operator_Taylor_Johnson"
metadata_store_client.create_alias(ALIAS_OPERATOR_TAYLOR, operator3)

# Create sample test stations
print("\n✅ Creating test stations...")
station1 = TestStation(name="TestStation_A1")
metadata_store_client.create_test_station(station1)
ALIAS_STATION_A1 = "V2_Station_A1"
metadata_store_client.create_alias(ALIAS_STATION_A1, station1)

station2 = TestStation(name="TestStation_B2")
metadata_store_client.create_test_station(station2)
ALIAS_STATION_B2 = "V2_Station_B2"
metadata_store_client.create_alias(ALIAS_STATION_B2, station2)

station3 = TestStation(name="TestStation_C3")
metadata_store_client.create_test_station(station3)
ALIAS_STATION_C3 = "V2_Station_C3"
metadata_store_client.create_alias(ALIAS_STATION_C3, station3)

# Create sample hardware items (test equipment)
print("\n✅ Creating hardware items...")
dmm = HardwareItem(manufacturer="NI", model="PXIe-4081", serial_number="DMM001")
metadata_store_client.create_hardware_item(dmm)
ALIAS_DMM = "V2_DMM_PXIe4081"
metadata_store_client.create_alias(ALIAS_DMM, dmm)

scope = HardwareItem(manufacturer="NI", model="PXIe-5171", serial_number="SCOPE001")
metadata_store_client.create_hardware_item(scope)
ALIAS_SCOPE = "V2_Scope_PXIe5171"
metadata_store_client.create_alias(ALIAS_SCOPE, scope)

# Create sample UUTs (Units Under Test - product definitions)
print("\n✅ Creating UUTs...")
power_supply_uut = Uut(model_name="PowerSupply v2.1", family="Power")
metadata_store_client.create_uut(power_supply_uut)
ALIAS_UUT_POWER_SUPPLY = "V2_UUT_PowerSupply_v2_1"
metadata_store_client.create_alias(ALIAS_UUT_POWER_SUPPLY, power_supply_uut)

amplifier_uut = Uut(model_name="Audio Amplifier v1.3", family="Audio")
metadata_store_client.create_uut(amplifier_uut)
ALIAS_UUT_AMPLIFIER = "V2_UUT_AudioAmplifier_v1_3"
metadata_store_client.create_alias(ALIAS_UUT_AMPLIFIER, amplifier_uut)

# Create sample software items
print("\n✅ Creating software items...")
python_env = SoftwareItem(product="Python", version="3.11.5")
metadata_store_client.create_software_item(python_env)
ALIAS_SOFTWARE_PYTHON = "V2_Software_Python_3_11_5"
metadata_store_client.create_alias(ALIAS_SOFTWARE_PYTHON, python_env)

pytest_env = SoftwareItem(product="pytest", version="7.4.0")
metadata_store_client.create_software_item(pytest_env)
ALIAS_SOFTWARE_PYTEST = "V2_Software_pytest_7_4_0"
metadata_store_client.create_alias(ALIAS_SOFTWARE_PYTEST, pytest_env)

nidaq_env = SoftwareItem(product="NI-DAQmx", version="23.3.0")
metadata_store_client.create_software_item(nidaq_env)
ALIAS_SOFTWARE_NIDAQ = "V2_Software_NI_DAQmx_23_3_0"
metadata_store_client.create_alias(ALIAS_SOFTWARE_NIDAQ, nidaq_env)

## Create UUT Instances

This setup is done for each test run. Each time we're going to run a test, we create instances of the specific UUT we are testing.

In [ ]:
print("\n✅ Creating UUT instances...")
ps_instance1 = UutInstance(uut_id=ALIAS_UUT_POWER_SUPPLY, serial_number="PS-2024-001")
metadata_store_client.create_uut_instance(ps_instance1)
ALIAS_UUT_INSTANCE_PS1 = "V2_UUT_Instance_PS_001"
metadata_store_client.create_alias(ALIAS_UUT_INSTANCE_PS1, ps_instance1)

ps_instance2 = UutInstance(uut_id=ALIAS_UUT_POWER_SUPPLY, serial_number="PS-2024-002")
metadata_store_client.create_uut_instance(ps_instance2)
ALIAS_UUT_INSTANCE_PS2 = "V2_UUT_Instance_PS_002"
metadata_store_client.create_alias(ALIAS_UUT_INSTANCE_PS2, ps_instance2)

amp_instance1 = UutInstance(uut_id=ALIAS_UUT_AMPLIFIER, serial_number="AMP-2024-001")
metadata_store_client.create_uut_instance(amp_instance1)
ALIAS_UUT_INSTANCE_AMP1 = "V2_UUT_Instance_AMP_001"
metadata_store_client.create_alias(ALIAS_UUT_INSTANCE_AMP1, amp_instance1)

## Create Test Results, Steps, and Measurements

The helper below creates one test result. For each step, it creates the step with its conditions, then creates a scalar measurement for each measurement name.

In V2, a test result starts as `RUNNING`. When all of its measurements are created, the helper marks it `DONE` and sets its outcome to `FAILED` if any measurement failed.

In [ ]:
import random

from nitypes.scalar import Scalar

from ni.datastore.data.v2 import Condition, Outcome, Status, Step, TestResult


def sample_value(measurement_name: str) -> tuple[float, str]:
    """Return a realistic value and units for a measurement name."""
    if "-12V" in measurement_name:
        return random.uniform(-12.2, -11.8), "V"
    if "5V" in measurement_name:
        return random.uniform(4.95, 5.05), "V"
    if "12V" in measurement_name:
        return random.uniform(11.8, 12.2), "V"
    if "Input Voltage" in measurement_name:
        return random.uniform(115, 125), "V"
    if "Supply Voltage" in measurement_name:
        return random.uniform(14.5, 15.5), "V"
    if "Standby Current" in measurement_name:
        return random.uniform(0.01, 0.05), "A"
    if "Current" in measurement_name:
        return random.uniform(0.5, 1.5), "A"
    if "DC Offset" in measurement_name:
        return random.uniform(-5, 5), "mV"
    if "Response" in measurement_name:
        return random.uniform(-1, 1), "dB"
    if "THD" in measurement_name and "Power" not in measurement_name:
        return random.uniform(0.001, 0.01), "%"
    if "Power" in measurement_name:
        return random.uniform(40, 150), "W"
    if "Ripple" in measurement_name:
        return random.uniform(10, 50), "mV"
    if "SNR" in measurement_name:
        return random.uniform(95, 105), "dB"
    return random.uniform(0, 100), ""


def create_sample_test(
    test_result: TestResult,
    steps: list[tuple[str, list[str]]],
    conditions: list[tuple[str, object, str, str]],
) -> tuple[str, int, int]:
    """Create a test result with steps, conditions, and measurements.

    Returns the test result ID, the number of measurements, and the number of conditions.
    """
    test_result_id = data_store_client.create_test_result(test_result)
    measurement_count = 0
    condition_count = 0
    any_failed = False
    for step_name, measurement_names in steps:
        step = Step(
            name=step_name,
            test_result_id=test_result_id,
            conditions=[
                Condition(name, value, condition_type=condition_type, units=units)
                for name, value, units, condition_type in conditions
            ],
        )
        step_id = data_store_client.create_step(step)
        condition_count += len(conditions)
        for measurement_name in measurement_names:
            value, units = sample_value(measurement_name)
            # Insert some random failures
            outcome = Outcome.PASSED if random.random() > 0.4 else Outcome.FAILED
            any_failed = any_failed or outcome == Outcome.FAILED
            data_store_client.create_measurement(
                measurement_name, Scalar(value, units), step_id, outcome=outcome
            )
            measurement_count += 1

    data_store_client.update_test_result(
        test_result_id,
        status=Status.DONE,
        outcome=Outcome.FAILED if any_failed else Outcome.PASSED,
    )
    return test_result_id, measurement_count, condition_count

In [ ]:
print("\n🔬 Creating sample test results with measurements...")

# Scenario 1: Power Supply Test. Use aliases instead of IDs for better readability.
print("📋 Creating Power Supply test...")
ps_test_result_id, ps_measurements, ps_conditions = create_sample_test(
    TestResult(
        name="Power Supply Test v2.1",
        uut_instance_id=ALIAS_UUT_INSTANCE_PS1,
        operator_id=ALIAS_OPERATOR_ALEX,
        test_station_id=ALIAS_STATION_A1,
        software_item_ids=[ALIAS_SOFTWARE_PYTHON],
        hardware_item_ids=[ALIAS_DMM, ALIAS_SCOPE],
        status=Status.RUNNING,
    ),
    steps=[
        ("Initialize", ["Configure DMM", "Set Load Conditions", "Warm-up Period"]),
        ("Input Voltage Test", ["Measure Input Voltage", "Measure Input Current", "Calculate Input Power"]),
        ("Output Voltage Test", ["Measure 5V Rail", "Measure 12V Rail", "Measure -12V Rail"]),
        ("Load Regulation", ["No Load Test", "25% Load Test", "50% Load Test", "75% Load Test", "Full Load Test"]),
        ("Ripple Measurement", ["5V Ripple", "12V Ripple", "-12V Ripple"]),
    ],
    conditions=[
        ("Ambient Temperature", random.uniform(21, 26), "°C", "Environment"),
        ("Barometric Pressure", random.uniform(995, 1015), "hPa", "Environment"),
        ("Line Frequency", random.uniform(59.9, 60.1), "Hz", "Environment"),
        # V2 conditions also support text values.
        ("Calibration Date", "2024-03-15", "", "Setup"),
        ("Test Fixture", "PS-TEST-001", "", "Setup"),
    ],
)
print(f"  📊 Created {ps_measurements} measurements and {ps_conditions} conditions")

# Scenario 2: Audio Amplifier Test
print("\n📋 Creating Audio Amplifier test...")
amp_test_result_id, amp_measurements, amp_conditions = create_sample_test(
    TestResult(
        name="Audio Amplifier Test v1.3",
        uut_instance_id=ALIAS_UUT_INSTANCE_AMP1,
        operator_id=ALIAS_OPERATOR_JORDAN,
        test_station_id=ALIAS_STATION_B2,
        software_item_ids=[ALIAS_SOFTWARE_PYTHON],
        hardware_item_ids=[ALIAS_DMM, ALIAS_SCOPE],
        status=Status.RUNNING,
    ),
    steps=[
        ("Power-On Test", ["Supply Voltage", "Standby Current", "Enable Signal"]),
        ("DC Offset", ["Left Channel DC Offset", "Right Channel DC Offset"]),
        ("Frequency Response", ["20Hz Response", "1kHz Response", "10kHz Response", "20kHz Response"]),
        ("THD+N Test", ["1kHz THD+N Left", "1kHz THD+N Right", "10kHz THD+N Left", "10kHz THD+N Right"]),
        ("Power Output", ["Max Power Left", "Max Power Right", "Power at 1% THD"]),
        ("Signal-to-Noise", ["SNR Left Channel", "SNR Right Channel"]),
    ],
    conditions=[
        ("Ambient Temperature", random.uniform(22, 25), "°C", "Environment"),
        ("Relative Humidity", random.uniform(45, 55), "%", "Environment"),
        ("Test Load", random.uniform(7.8, 8.2), "Ω", "Setup"),
    ],
)
print(f"  📊 Created {amp_measurements} measurements and {amp_conditions} conditions")

# Scenario 3: Additional Power Supply Test (second unit, quick verification)
print("\n📋 Creating second Power Supply test...")
ps2_test_result_id, ps2_measurements, ps2_conditions = create_sample_test(
    TestResult(
        name="Power Supply Test v2.1",
        uut_instance_id=ALIAS_UUT_INSTANCE_PS2,
        operator_id=ALIAS_OPERATOR_TAYLOR,
        test_station_id=ALIAS_STATION_C3,
        software_item_ids=[ALIAS_SOFTWARE_PYTHON],
        hardware_item_ids=[ALIAS_DMM, ALIAS_SCOPE],
        status=Status.RUNNING,
    ),
    steps=[("Quick Verification", ["Output 5V", "Output 12V", "Load Current"])],
    conditions=[("Room Temperature", random.uniform(20, 24), "°C", "Environment")],
)
print(f"  📊 Created {ps2_measurements} measurements and {ps2_conditions} conditions")

total_measurements = ps_measurements + amp_measurements + ps2_measurements
total_conditions = ps_conditions + amp_conditions + ps2_conditions
assert total_measurements == 38

print("\n🎯 Sample data creation complete!")
print("📊 Summary:")
print("  - Test Results: 3")
print("  - Steps: 12")
print(f"  - Measurements: {total_measurements}")
print(f"  - Conditions: {total_conditions}")
print("\n✅ Ready to run query_measurements.ipynb!")

## Clean up

In [ ]:
data_store_client.close()
metadata_store_client.close()

# Perform example-specific cleanup. This is not needed when writing production code.
if grpc_channel is not None:
    grpc_channel.close()

## Summary

✅ **Sample data has been successfully created!**

You can now run **`query_measurements.ipynb`** to learn how to query test results, steps, measurements, and conditions with the V2 API.

To query metadata, such as operators and hardware, use `../../query/query_metadata.ipynb`. The Metadata API is the same in V1 and V2.